### Q1. Running Elastic

In [1]:
!curl localhost:9200

{
  "name" : "75746bb9537d",
  "cluster_name" : "docker-cluster",
  "cluster_uuid" : "0D1p2DIoTAamfqRAJkC3Vw",
  "version" : {
    "number" : "8.4.3",
    "build_flavor" : "default",
    "build_type" : "docker",
    "build_hash" : "42f05b9372a9a4a470db3b52817899b99a76ee73",
    "build_date" : "2022-10-04T07:17:24.662462378Z",
    "build_snapshot" : false,
    "lucene_version" : "9.3.0",
    "minimum_wire_compatibility_version" : "7.17.0",
    "minimum_index_compatibility_version" : "7.0.0"
  },
  "tagline" : "You Know, for Search"
}


What's the version.build_hash value?

version.build_hash = "42f05b9372a9a4a470db3b52817899b99a76ee73"

### Getting the data

In [2]:
import requests

docs_url = 'https://github.com/DataTalksClub/llm-zoomcamp/blob/main/01-intro/documents.json?raw=1'
docs_response = requests.get(docs_url)
documents_raw = docs_response.json()

documents = []

for course in documents_raw:
    course_name = course['course']

    for doc in course['documents']:
        doc['course'] = course_name
        documents.append(doc)

### Q2. Indexing the data

In [5]:
from elasticsearch import Elasticsearch

es_client = Elasticsearch('http://localhost:9200')

index_settings = {
    "settings": {
        "number_of_shards": 1,
        "number_of_replicas": 0
    },
    "mappings": {
        "properties": {
            "text": {"type": "text"},
            "section": {"type": "text"},
            "question": {"type": "text"},
            "course": {"type": "keyword"}
        }
    }
}

index_name = "homework-questions"

es_client.indices.create(index=index_name, body=index_settings)

from tqdm.auto import tqdm

for doc in tqdm(documents):
    es_client.index(index=index_name, document=doc)

  0%|          | 0/948 [00:00<?, ?it/s]

Which function do you use for adding your data to elastic?

index

### Q3. Searching

In [10]:
def elastic_search(query):
    search_query = {
        "query": {
            "bool": {
                "must": {
                    "multi_match": {
                        "query": query,
                        "fields": ["question^4", "text"],
                        "type": "best_fields"
                    }
                }
            }
        }
    }

    response = es_client.search(index=index_name, body=search_query)

    results = []

    for hit in response['hits']['hits']:
        results.append({"doc": hit['_source'], "score": hit['_score']})

    return results

query = 'How do execute a command on a Kubernetes pod?'
results = elastic_search(query)
print(max([r["score"] for r in results]))

44.50556


What's the score for the top ranking result?

44.50

### Q4. Filtering

In [12]:
def elastic_search(query):
    search_query = {
        "size": 3,
        "query": {
            "bool": {
                "must": {
                    "multi_match": {
                        "query": query,
                        "fields": ["question^4", "text"],
                        "type": "best_fields"
                    }
                },
                "filter": {
                    "term": {
                        "course": "machine-learning-zoomcamp"
                    }
                }
            }
        }
    }

    response = es_client.search(index=index_name, body=search_query)

    results = []

    for hit in response['hits']['hits']:
        results.append({"doc": hit['_source'], "score": hit['_score']})

    return results

query = 'How do copy a file to a Docker container?'
results = elastic_search(query)
print([r["doc"]["question"] for r in results])

['How do I debug a docker container?', 'How do I copy files from my local machine to docker container?', 'How do I copy files from a different folder into docker container’s working directory?']


Return 3 results. What's the 3rd question returned by the search engine?

'How do I copy files from a different folder into docker container’s working directory?'

### Q5. Building a prompt

In [16]:
def build_prompt(query, search_results):
    context_template = """
Q: {question}
A: {text}
""".strip()

    context = ""

    for doc in search_results:
        context = context + "\n\n" + context_template.format(question=doc['question'], text=doc['text'])

    prompt_template = """
You're a course teaching assistant. Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.

QUESTION: {question}

CONTEXT:
{context}
""".strip()


    prompt = prompt_template.format(question=query, context=context.strip()).strip()
    return prompt

prompt = build_prompt(query, [r["doc"] for r in results])
print(len(prompt))

1446


What's the length of the resulting prompt? (use the len function)

1446

### Q6. Tokens

In [15]:
!pip install tiktoken

  Using cached tiktoken-0.9.0-cp310-cp310-macosx_11_0_arm64.whl.metadata (6.7 kB)
  Using cached regex-2024.11.6-cp310-cp310-macosx_11_0_arm64.whl.metadata (40 kB)
Using cached tiktoken-0.9.0-cp310-cp310-macosx_11_0_arm64.whl (1.0 MB)
Using cached regex-2024.11.6-cp310-cp310-macosx_11_0_arm64.whl (284 kB)

[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


In [17]:
import tiktoken

encoding = tiktoken.encoding_for_model("gpt-4o")

tokens = encoding.encode(prompt)
print(len(tokens))

320


How many tokens does our prompt have?

320

### Bonus: generating the answer (ungraded)

In [20]:
from openai import OpenAI

from dotenv import load_dotenv
from pathlib import Path
dotenv_path = Path().resolve().parent.parent.parent / ".env"
load_dotenv(dotenv_path)
client = OpenAI()

def llm(prompt):
    response = client.chat.completions.create(
        model='gpt-4o',
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content

response = llm(prompt)
print(response)

/Users/vladimirkornyshev/lang/github.com/gnuzzz/llm-zoomcamp/.env
To copy a file from your local machine to a Docker container, you can use the `docker cp` command. The basic syntax is as follows:

```bash
docker cp /path/to/local/file_or_directory container_id:/path/in/container
```

This command allows you to copy files or directories from your local machine into a running Docker container.


What's the response?

To copy a file from your local machine to a Docker container, you can use the `docker cp` command. The basic syntax is as follows:

```bash
docker cp /path/to/local/file_or_directory container_id:/path/in/container
```

This command allows you to copy files or directories from your local machine into a running Docker container.

### Bonus: calculating the costs (ungraded)

In [25]:
tokens_per_request = 150
tokens_per_response = 250
num_requests = 1000

def cost(input_len, output_len):
    input_price = 0.005
    output_price = 0.015
    input_cost = input_len / 1000 * input_price
    output_cost = output_len / 1000 * output_price
    return input_cost + output_cost

cost_all = cost(150, 250) * 1000
print(cost_all)

response_tokens = encoding.encode(response)
print(len(response_tokens))
cost_q6_q7 = cost(len(tokens), len(response_tokens))
print(cost_q6_q7)

4.5
70
0.0026500000000000004
